# 5-Fold Ensemble Inference - Blind Test Dataset

**Pipeline Logic:**
1. **Segmentation (Model 2a):** Predict lesion mask on normalized image
2. **Binarize & Count:** Binarize mask and count white pixels
3. **Normal Filter:** If mask is mostly empty → P(Normal) = 1.0
4. **Mask RAW Image:** Apply binary mask to raw (unnormalized) image
5. **Normalize Masked Image:** Normalize the masked image for classifier
6. **Classification with TTA:** Classify using both original and horizontally flipped images
7. **Confidence-Weighted Ensemble:** Average probabilities across 5 folds (Soft Voting)

**Output:** `submission_results.xlsx` with columns `US` (filename) and `LABEL` (0/1/2)

## 0. Load Test Data

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SET DATA PATHS (DATA ALREADY AVAILABLE LOCALLY)
# ═══════════════════════════════════════════════════════════════════════════
import os
import glob
import pandas as pd

# Data paths
DATA_BASE_PATH = '../input/datasets/samansimpich/test-1'
TEST_IMAGE_FOLDER = os.path.join(DATA_BASE_PATH, 'test_images')
TEST_METADATA_PATH = os.path.join(DATA_BASE_PATH, 'test_metadata.xlsx')

print("Test data location:")
print(f"  Base path: {DATA_BASE_PATH}")
print(f"  Images folder: {TEST_IMAGE_FOLDER}")
print(f"  Metadata file: {TEST_METADATA_PATH}")

# Verify files exist
print(f"\n✓ Images folder exists: {os.path.exists(TEST_IMAGE_FOLDER)}")
print(f"✓ Metadata file exists: {os.path.exists(TEST_METADATA_PATH)}")

# Count images
if os.path.exists(TEST_IMAGE_FOLDER):
    image_files = glob.glob(os.path.join(TEST_IMAGE_FOLDER, '*.png')) + \
                  glob.glob(os.path.join(TEST_IMAGE_FOLDER, '*.jpg'))
    print(f"✓ Found {len(image_files)} images")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD TEST METADATA
# ═══════════════════════════════════════════════════════════════════════════

# Load the metadata
test_metadata = pd.read_excel(TEST_METADATA_PATH)

print(f"✓ Loaded metadata from: {TEST_METADATA_PATH}")
print(f"  Shape: {test_metadata.shape}")
print(f"  Columns: {list(test_metadata.columns)}")

# Display first few rows
print("\nFirst 5 rows:")
display(test_metadata.head())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# IDENTIFY FILENAME COLUMN
# ═══════════════════════════════════════════════════════════════════════════

# Try to identify the filename column
possible_filename_cols = ['US', 'filename', 'image_filename', 'Filename', 'Image', 'image', 'file']
FILENAME_COL = None

for col in possible_filename_cols:
    if col in test_metadata.columns:
        FILENAME_COL = col
        break

# If not found, use the first column
if FILENAME_COL is None:
    FILENAME_COL = test_metadata.columns[0]

print(f"✓ Using filename column: '{FILENAME_COL}'")
print(f"  Sample values: {test_metadata[FILENAME_COL].head().tolist()}")

# Get list of filenames
test_filenames = test_metadata[FILENAME_COL].tolist()
print(f"\n✓ Total test images: {len(test_filenames)}")

## 1. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🎯 ENSEMBLE CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

# Model Architecture (must match the trained models)
MODEL_2A_ARCH = 'EfficientUNet'  # Segmentation model architecture
MODEL_2B_ARCH = 'ResNet18'       # Classification model architecture

# Path to K-Fold Models
MODELS_BASE_PATH = '/kaggle/input/models/samansimpich/aaib-k-fold-models/tensorflow1/default/1/'

# Pipeline Parameters
MASK_THRESHOLD = 50              # Minimum white pixels to consider as "lesion present"
MASK_BINARIZE_THRESHOLD = 0.4    # Probability threshold for binarizing segmentation output

# Data Configuration
IMG_SIZE = 224
BATCH_SIZE = 1  # MUST be 1 for per-sample probability tracking

# Number of Folds
NUM_FOLDS = 5

# Classification dropout rate (must match training)
DROPOUT_RATE = 0.3

# Class names
CLASS_NAMES = ['Benign', 'Malignant', 'Normal']

print("="*70)
print("🔬 5-FOLD ENSEMBLE INFERENCE CONFIGURATION (BLIND TEST)")
print("="*70)
print(f"Segmentation Model (2a): {MODEL_2A_ARCH}")
print(f"Classification Model (2b): {MODEL_2B_ARCH}")
print(f"Test Images: {len(test_filenames)}")
print(f"Mask Threshold: {MASK_THRESHOLD} pixels")
print(f"Mask Binarize Threshold: {MASK_BINARIZE_THRESHOLD}")
print(f"TTA: Enabled (Horizontal Flip)")
print(f"Ensemble Method: Soft Voting (Probability Averaging)")
print("="*70)

## 2. Imports & Setup

In [ ]:
# Install dependencies if needed
!pip install segmentation-models-pytorch albumentations openpyxl -q

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Device Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("✓ All imports loaded successfully!")

## 3. Model Factory Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MODEL 2A: SEGMENTATION (using segmentation_models_pytorch)
# ═══════════════════════════════════════════════════════════════════════════

def get_model_2a(arch_name):
    """
    Create a segmentation model for Task 2a.
    
    Args:
        arch_name: 'ResNetUNet', 'EfficientUNet', or 'AttentionUNet'
    
    Returns:
        torch.nn.Module with sigmoid output
    """
    if arch_name == 'ResNetUNet':
        base_model = smp.Unet(
            encoder_name='resnet34',
            encoder_weights=None,
            in_channels=1,
            classes=1,
            activation=None
        )
    elif arch_name == 'EfficientUNet':
        base_model = smp.Unet(
            encoder_name='efficientnet-b0',
            encoder_weights=None,
            in_channels=1,
            classes=1,
            activation=None
        )
    elif arch_name == 'AttentionUNet':
        base_model = smp.UnetPlusPlus(
            encoder_name='resnet34',
            encoder_weights=None,
            in_channels=1,
            classes=1,
            activation=None
        )
    else:
        raise ValueError(f"Unknown segmentation architecture: {arch_name}")
    
    # Wrapper to apply sigmoid
    class SegmentationModelWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        
        def forward(self, x):
            return torch.sigmoid(self.model(x))
    
    return SegmentationModelWrapper(base_model)


# ═══════════════════════════════════════════════════════════════════════════
# MODEL 2B: CLASSIFICATION (using torchvision)
# ═══════════════════════════════════════════════════════════════════════════

def get_model_2b(arch_name, dropout_rate=0.3):
    """
    Create a binary classification model for Task 2b.
    
    Args:
        arch_name: 'EfficientNetB0', 'ResNet18', or 'MobileNetV3'
        dropout_rate: Dropout rate (must match training)
    
    Returns:
        torch.nn.Module with 2-class output
    """
    if arch_name == 'EfficientNetB0':
        model = models.efficientnet_b0(weights=None)
        model.features[0][0] = nn.Conv2d(
            1, 32, kernel_size=3, stride=2, padding=1, bias=False
        )
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 2)
        )
    
    elif arch_name == 'ResNet18':
        model = models.resnet18(weights=None)
        model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 2)
        )
    
    elif arch_name == 'MobileNetV3':
        model = models.mobilenet_v3_small(weights=None)
        model.features[0][0] = nn.Conv2d(
            1, 16, kernel_size=3, stride=2, padding=1, bias=False
        )
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, 2)
        )
    
    else:
        raise ValueError(f"Unknown classification architecture: {arch_name}")
    
    return model


print("✓ Model factory functions defined")

## 4. Test Inference Dataset (No Labels Required)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TEST INFERENCE DATASET - NO LABELS REQUIRED
# Returns both normalized (for Model 2a) and raw (for masking) images
# ═══════════════════════════════════════════════════════════════════════════

class TestInferenceDataset(Dataset):
    """
    Dataset for blind test inference.
    Does NOT require ground truth labels.
    
    Args:
        filenames: List of image filenames
        image_dir: Directory containing the images
        img_size: Target image size (default: 224)
    """
    def __init__(self, filenames, image_dir, img_size=224):
        self.filenames = filenames
        self.image_dir = image_dir
        self.img_size = img_size
        
        # Transforms for normalized image (for Model 2a - Segmentation)
        self.transform_normalized = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2()
        ])
        
        # Transforms for raw image (for masking, kept in [0,1] range)
        self.transform_raw = A.Compose([
            A.Resize(img_size, img_size),
            ToTensorV2()
        ])
        
        # Verify images exist
        missing = []
        for fn in self.filenames:
            # Try with and without extension
            img_path = os.path.join(self.image_dir, fn)
            if not os.path.exists(img_path):
                # Try adding .png extension
                if not os.path.exists(img_path + '.png'):
                    missing.append(fn)
        
        if missing:
            print(f"⚠️  Warning: {len(missing)} images not found")
            if len(missing) <= 5:
                print(f"   Missing: {missing}")
        
        print(f"✓ TestInferenceDataset: {len(self.filenames)} images")
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        
        # Build image path (handle with/without extension)
        img_path = os.path.join(self.image_dir, img_name)
        if not os.path.exists(img_path):
            # Try adding .png extension
            img_path = os.path.join(self.image_dir, img_name + '.png')
        
        # Load image as grayscale
        image = np.array(Image.open(img_path).convert("L"))
        
        # Normalized image (for Model 2a - Segmentation)
        aug_normalized = self.transform_normalized(image=image)
        image_normalized = aug_normalized['image']
        
        # Raw image (for masking, [0,1] range)
        aug_raw = self.transform_raw(image=image)
        image_raw = aug_raw['image'].float() / 255.0
        
        # Add channel dimension if needed
        if image_normalized.ndim == 2:
            image_normalized = image_normalized.unsqueeze(0)
        if image_raw.ndim == 2:
            image_raw = image_raw.unsqueeze(0)
        
        return image_normalized, image_raw, img_name


print("✓ TestInferenceDataset class defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CREATE TEST DATALOADER
# ═══════════════════════════════════════════════════════════════════════════

print(f"{'='*60}")
print(f"LOADING TEST DATA")
print(f"{'='*60}")

# Create dataset and loader
test_dataset = TestInferenceDataset(
    filenames=test_filenames,
    image_dir=TEST_IMAGE_FOLDER,
    img_size=IMG_SIZE
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Maintain order for submission
    num_workers=2
)

print(f"✓ DataLoader created with {len(test_loader)} batches")

## 5. Find Model Paths for All Folds

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FIND MODEL PATHS FOR ALL 5 FOLDS
# Model naming convention: best_model_{TASK}_{ARCH}_f{FOLD}-{score}.pth
# ═══════════════════════════════════════════════════════════════════════════

def find_model_path(models_dir, task, arch, fold):
    """
    Find the model path matching the pattern using recursive search.
    Handles varying score formats and unknown sub-directory structures.
    """
    # The file pattern itself
    pattern = f"best_model_{task}_{arch}_f{fold}-*.pth"
    
    # Add "**" to allow glob to search through any intermediate folders (or no folders)
    search_path = os.path.join(models_dir, "**", pattern)
    
    # Enable recursive=True so '**' works correctly
    matches = glob.glob(search_path, recursive=True)
    
    if not matches:
        raise FileNotFoundError(f"No model found matching: {search_path}")
    
    # Return the first match 
    return matches[0]

# Find all model paths
print("="*70)
print("LOCATING 5-FOLD MODELS")
print("="*70)

model_paths = {
    '2a': {},
    '2b': {}
}

for fold in range(NUM_FOLDS):
    # Find Model 2a (Segmentation)
    path_2a = find_model_path(MODELS_BASE_PATH, '2a', MODEL_2A_ARCH, fold)
    model_paths['2a'][fold] = path_2a
    
    # Find Model 2b (Classification)
    path_2b = find_model_path(MODELS_BASE_PATH, '2b', MODEL_2B_ARCH, fold)
    model_paths['2b'][fold] = path_2b
    
    print(f"\nFold {fold}:")
    print(f"  2a: {os.path.basename(path_2a)}")
    print(f"  2b: {os.path.basename(path_2b)}")

print("\n" + "="*70)
print("✓ All 10 models located successfully")
print("="*70)

## 6. 5-Fold Ensemble Inference with TTA

**Memory Management Strategy:**
- Load only ONE fold's models at a time
- Run inference on entire test set
- Store probabilities, delete models, clear GPU cache
- Repeat for all 5 folds

**TTA (Test-Time Augmentation):**
- Original image + Horizontally flipped image
- Average softmax probabilities

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5-FOLD ENSEMBLE INFERENCE WITH TTA AND MEMORY MANAGEMENT
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("🚀 STARTING 5-FOLD ENSEMBLE INFERENCE (BLIND TEST)")
print("="*70)
print("\n⚠️  Memory-Safe Mode: Loading one fold at a time")
print("📊 Probability Storage: [P(Benign), P(Malignant), P(Normal)]")
print("🔄 TTA: Horizontal flip averaging for classification")
print("="*70 + "\n")

# Normalization parameters (same as training)
NORM_MEAN = torch.tensor([0.5]).view(1, 1, 1, 1).to(DEVICE)
NORM_STD = torch.tensor([0.5]).view(1, 1, 1, 1).to(DEVICE)

# Storage for all folds' probabilities
# Shape will be: [num_folds, num_samples, 3] -> [5, N, 3]
num_samples = len(test_dataset)
all_fold_probs = np.zeros((NUM_FOLDS, num_samples, 3), dtype=np.float32)

# Storage for filenames (collected once)
filenames_list = []
filenames_collected = False

# ═══════════════════════════════════════════════════════════════════════════
# MAIN FOLD LOOP - Memory Safe
# ═══════════════════════════════════════════════════════════════════════════

for fold in range(NUM_FOLDS):
    print(f"\n{'─'*70}")
    print(f"📂 FOLD {fold} / {NUM_FOLDS-1}")
    print(f"{'─'*70}")
    
    # ═══════════════════════════════════════════════════════════════════════
    # STEP A: Load Models for This Fold
    # ═══════════════════════════════════════════════════════════════════════
    print(f"  Loading models...")
    
    # Load Model 2a (Segmentation)
    model_2a = get_model_2a(MODEL_2A_ARCH).to(DEVICE)
    model_2a.load_state_dict(torch.load(model_paths['2a'][fold], map_location=DEVICE))
    model_2a.eval()
    
    # Load Model 2b (Classification)
    model_2b = get_model_2b(MODEL_2B_ARCH, dropout_rate=DROPOUT_RATE).to(DEVICE)
    model_2b.load_state_dict(torch.load(model_paths['2b'][fold], map_location=DEVICE))
    model_2b.eval()
    
    print(f"  ✓ Model 2a: {os.path.basename(model_paths['2a'][fold])}")
    print(f"  ✓ Model 2b: {os.path.basename(model_paths['2b'][fold])}")
    
    # ═══════════════════════════════════════════════════════════════════════
    # STEP B: Run Inference on Entire Test Set
    # ═══════════════════════════════════════════════════════════════════════
    print(f"  Running inference with TTA...")
    
    sample_idx = 0
    
    with torch.no_grad():
        for images_normalized, images_raw, img_names in tqdm(
            test_loader, desc=f"  Fold {fold} Inference", leave=False
        ):
            # Move to device
            images_normalized = images_normalized.to(DEVICE)
            images_raw = images_raw.to(DEVICE)
            img_name = img_names[0]
            
            # Collect filenames only on first fold
            if not filenames_collected:
                filenames_list.append(img_name)
            
            # ═══════════════════════════════════════════════════════════════
            # STEP 1: Segmentation (Model 2a) - Uses NORMALIZED image
            # ═══════════════════════════════════════════════════════════════
            pred_mask_prob = model_2a(images_normalized)  # [B, 1, H, W]
            pred_mask_binary = (pred_mask_prob > MASK_BINARIZE_THRESHOLD).float()
            
            # Count white pixels in the predicted mask
            pixel_count = pred_mask_binary.sum().item()
            
            # ═══════════════════════════════════════════════════════════════
            # STEP 2: Normal Filter
            # ═══════════════════════════════════════════════════════════════
            if pixel_count < MASK_THRESHOLD:
                # No significant lesion detected → 100% Normal
                fold_probs = [0.0, 0.0, 1.0]  # [P(Benign), P(Malignant), P(Normal)]
            else:
                # ═══════════════════════════════════════════════════════════
                # STEP 3a: Apply mask to RAW (unnormalized) image
                # ═══════════════════════════════════════════════════════════
                masked_image_raw = images_raw * pred_mask_binary  # [0, 1] range
                
                # ═══════════════════════════════════════════════════════════
                # STEP 3b: Normalize the masked image (AFTER masking)
                # ═══════════════════════════════════════════════════════════
                masked_image_normalized = (masked_image_raw - NORM_MEAN) / NORM_STD
                
                # ═══════════════════════════════════════════════════════════
                # STEP 3c: Classification with TTA (Horizontal Flip)
                # ═══════════════════════════════════════════════════════════
                
                # Original image prediction
                output_orig = model_2b(masked_image_normalized)
                probs_orig = F.softmax(output_orig, dim=1)  # [B, 2]
                
                # Horizontally flipped image prediction (TTA)
                masked_image_flipped = torch.flip(masked_image_normalized, dims=[3])  # Flip along width
                output_flipped = model_2b(masked_image_flipped)
                probs_flipped = F.softmax(output_flipped, dim=1)  # [B, 2]
                
                # Average TTA probabilities
                avg_probs = (probs_orig + probs_flipped) / 2.0  # [B, 2]
                avg_probs = avg_probs.squeeze(0).cpu().numpy()  # [2]
                
                # Store as [P(Benign), P(Malignant), P(Normal)]
                # Classifier outputs [P(Benign), P(Malignant)], P(Normal) = 0
                fold_probs = [avg_probs[0], avg_probs[1], 0.0]
            
            # Store probabilities for this fold
            all_fold_probs[fold, sample_idx, :] = fold_probs
            sample_idx += 1
    
    filenames_collected = True  # Only collect filenames on first fold
    
    # ═══════════════════════════════════════════════════════════════════════
    # STEP C: Cleanup Memory - CRUCIAL!
    # ═══════════════════════════════════════════════════════════════════════
    print(f"  Cleaning up GPU memory...")
    del model_2a, model_2b
    torch.cuda.empty_cache()
    print(f"  ✓ Fold {fold} complete - models released")

print("\n" + "="*70)
print("✓ ALL 5 FOLDS PROCESSED SUCCESSFULLY")
print("="*70)
print(f"\nProbability array shape: {all_fold_probs.shape}")
print(f"Test samples processed: {len(filenames_list)}")

## 7. Ensemble Predictions (Soft Voting)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIDENCE-WEIGHTED ENSEMBLE (SOFT VOTING)
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("🎯 ENSEMBLE AGGREGATION (SOFT VOTING)")
print("="*70)

# Average probabilities across all 5 folds
# Shape: [num_samples, 3]
ensemble_probs = np.mean(all_fold_probs, axis=0)

print(f"\nEnsemble probability shape: {ensemble_probs.shape}")
print(f"\nSample ensemble probabilities (first 10 samples):")
print(f"{'Sample':<8} {'P(Benign)':<12} {'P(Malignant)':<14} {'P(Normal)':<12} {'Prediction'}")
print("-" * 65)

for i in range(min(10, len(ensemble_probs))):
    probs = ensemble_probs[i]
    pred = np.argmax(probs)
    print(f"{i:<8} {probs[0]:<12.4f} {probs[1]:<14.4f} {probs[2]:<12.4f} {CLASS_NAMES[pred]}")

# Final ensemble predictions (argmax of averaged probabilities)
y_pred_ensemble = np.argmax(ensemble_probs, axis=1)

print(f"\n✓ Ensemble predictions computed for {len(y_pred_ensemble)} samples")

# Distribution of predictions
print("\nPrediction Distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (y_pred_ensemble == i).sum()
    pct = count / len(y_pred_ensemble) * 100
    print(f"  {name}: {count} ({pct:.1f}%)")

## 8. Generate Submission File

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GENERATE SUBMISSION FILE
# Format: US (filename), LABEL (0, 1, or 2)
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("💾 GENERATING SUBMISSION FILE")
print("="*70)

# Create submission dataframe
submission_df = pd.DataFrame({
    'US': filenames_list,
    'LABEL': y_pred_ensemble
})

# Verify the submission
print(f"\nSubmission DataFrame:")
print(f"  Columns: {list(submission_df.columns)}")
print(f"  Shape: {submission_df.shape}")
print(f"\nFirst 10 rows:")
display(submission_df.head(10))

print(f"\nLabel Distribution:")
for label in [0, 1, 2]:
    count = (submission_df['LABEL'] == label).sum()
    pct = count / len(submission_df) * 100
    print(f"  {label} ({CLASS_NAMES[label]}): {count} ({pct:.1f}%)")

# Save to Excel
output_filename = 'submission_results.xlsx'
submission_df.to_excel(output_filename, index=False)

print(f"\n" + "="*70)
print(f"✅ SUBMISSION FILE SAVED: {output_filename}")
print(f"="*70)
print(f"\nFile contains:")
print(f"  - {len(submission_df)} predictions")
print(f"  - Column 'US': Image filenames")
print(f"  - Column 'LABEL': Predicted class (0=Benign, 1=Malignant, 2=Normal)")

## 9. Save Detailed Results (Optional)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SAVE DETAILED RESULTS (OPTIONAL - FOR ANALYSIS)
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("💾 SAVING DETAILED RESULTS")
print("="*70)

# Get confidence (max probability) for each prediction
confidences = np.max(ensemble_probs, axis=1)

# Create detailed results dataframe
detailed_results_df = pd.DataFrame({
    'US': filenames_list,
    'LABEL': y_pred_ensemble,
    'predicted_class': [CLASS_NAMES[i] for i in y_pred_ensemble],
    'confidence': confidences,
    'prob_benign': ensemble_probs[:, 0],
    'prob_malignant': ensemble_probs[:, 1],
    'prob_normal': ensemble_probs[:, 2]
})

# Add per-fold probabilities
for fold in range(NUM_FOLDS):
    detailed_results_df[f'fold_{fold}_prob_benign'] = all_fold_probs[fold, :, 0]
    detailed_results_df[f'fold_{fold}_prob_malignant'] = all_fold_probs[fold, :, 1]
    detailed_results_df[f'fold_{fold}_prob_normal'] = all_fold_probs[fold, :, 2]

# Save detailed results to CSV
detailed_output_filename = f'detailed_predictions_{MODEL_2A_ARCH}_{MODEL_2B_ARCH}_blind_test.csv'
detailed_results_df.to_csv(detailed_output_filename, index=False)
print(f"\n✓ Detailed predictions saved to: {detailed_output_filename}")

# Confidence statistics
print(f"\nConfidence Statistics:")
print(f"  Mean: {confidences.mean():.4f}")
print(f"  Std:  {confidences.std():.4f}")
print(f"  Min:  {confidences.min():.4f}")
print(f"  Max:  {confidences.max():.4f}")

print("\n" + "="*70)
print("✅ BLIND TEST INFERENCE COMPLETE")
print("="*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VERIFY SUBMISSION FILE
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("🔍 VERIFYING SUBMISSION FILE")
print("="*70)

# Read back the submission file to verify
verify_df = pd.read_excel('submission_results.xlsx')

print(f"\n✓ File loaded successfully")
print(f"  Shape: {verify_df.shape}")
print(f"  Columns: {list(verify_df.columns)}")
print(f"\nColumn types:")
print(verify_df.dtypes)

print(f"\nSample rows:")
display(verify_df.head())

# Check for any issues
issues = []
if 'US' not in verify_df.columns:
    issues.append("Missing 'US' column")
if 'LABEL' not in verify_df.columns:
    issues.append("Missing 'LABEL' column")
if verify_df['LABEL'].isnull().any():
    issues.append(f"Found {verify_df['LABEL'].isnull().sum()} null LABEL values")
if not set(verify_df['LABEL'].unique()).issubset({0, 1, 2}):
    issues.append(f"Invalid LABEL values found: {verify_df['LABEL'].unique()}")

if issues:
    print("\n⚠️  Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("\n✅ Submission file is valid!")
    print(f"   - {len(verify_df)} predictions")
    print(f"   - All labels are in {{0, 1, 2}}")
    print(f"   - No missing values")